## Master pipeline: data → model → predictions (tennis)

One call per cell into `src/pipeline/` (how it works: `docs/studies/master_pipeline/README.md`). As in the UFC master script, **the most frequent cells come first**; within each section, run the cells top to bottom.

| Section | When | What |
|---|---|---|
| **Setup**, **Status** | every session | imports and settings (incl. `MODEL`, `SPEND_TOKENS`, `STAKE_RULE`); where everything stands |
| **A. Bet day** | each betting day | morning refresh → Pinnacle's listing → today's matches → matchups CSV → **the data check** → predict → **the bet sheet**; reprice when prices move; grade later |
| **B. Data update** | weekly, and before a rebuild | results and box scores, serve speeds, timeline sources, odds history, the shared tables' rebuild |
| **C. Rebuild and retrain** | new season, or after accepted data | build a model instance, its odds input and 57 inputs, the season fits, checkpoint, the replay proof; Thinking's season fit |

**The data is stored and pulled differently from UFC.** A Windows task downloads results, box scores and serve speeds every day at 07:30. A prediction run rebuilds that day's tables in a sandbox, then resumes each feature family from a saved checkpoint, so nothing here edits the shared tables except B6.

**`MODEL`** (set in Setup):
* `"tabpfn35"`: the current best model alone. That is local TabPFN-3.5 on 57 inputs, seeds 0–15, plus stage 2, the serve-speed adjustment with its walk-forward calibration slope.
* `"tabpfn35_thinking"` (since 20 Oct 2026, [decision](../docs/decisions/2026-10-20-thinking-beside-local.md)): the same model **and** TabPFN-3.5-Thinking beside it, a paid hosted model on the same 57 inputs and training rows whose probabilities are used as the API gives them (not calibrated), plus stage 2's serve-speed nudge for men (from the evening of 20 Oct 2026).
  * The bet sheet shows the non-thinking, the thinking and the ensemble chance side by side. It decides on the ensemble: the 50/50 average in log-odds, plus the five-pattern adjustment.
  * It needs `SPEND_TOKENS = True`. Each A4 run makes one paid call (about 32k tokens). The first run on a new instance also makes its season fit (about 180k tokens and 8 minutes).
  * If the Thinking call fails, the sheet is the local model's alone and says so.

**Where the outputs land:**
* **the bet sheet** (A5, and a new one after each A6 reprice): `predictions/<day>/bet_sheet_<made>.xlsx`, shown under A5 too. Its sheets: Bet sheet (what to bet, at what odds, how much, and why), Matches (every figure), Not predicted, Run, How to read. Every price on it is in American odds.
* the machine record behind each sheet, and the daily engine's raw archive of the run: `predictions/<day>/records/`
* the data check (A3b): `data/studies/master_pipeline/data_check/` (each check's table and gaps; its Stage 0 cache)
* prediction runs: `data/studies/daily_prediction/runs/live_<date>_<time>/` (a Thinking run's paid call and its receipts in its `thinking/` folder)
* every step's log and receipt: `data/studies/master_pipeline/`
* Pinnacle's pulls: `data/raw/pinnacle/` and `data/dump/odds/pinnacle/pulls/`

`DRY_RUN = True` prints every command instead of running it.

In [ ]:
# SETUP: run first, once per kernel (kernel: the `tennis` conda env). Everything below is a call into src/pipeline/.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "python.sh").exists() and ROOT != ROOT.parent:     # the repository root, from scripts/ or anywhere
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import src.pipeline as tp

MODEL = "tabpfn35"       # "tabpfn35": the current best model alone (local TabPFN-3.5 + calibrated stage 2)
                         # "tabpfn35_thinking": that model AND TabPFN-3.5-Thinking (hosted, paid, uncalibrated) side by
                         #   side; the bet sheet decides on their 50/50 average. Needs SPEND_TOKENS = True
SPEND_TOKENS = False     # True: allow the paid Thinking calls of "tabpfn35_thinking" (each A4 run: one call, ~32k tokens;
                         #   a new instance's first Thinking run also makes its season fit, ~180k tokens)
DRY_RUN = False          # True: every step prints its command and runs nothing
HOURS_AHEAD = 36         # matches starting within this many hours are candidates
STALE_MINUTES = 30       # a quote older than this blocks a bet on the bet sheet (reprice first: A6)
STAKE_RULE = "owner_8oct"  # the bet sheet's stake: your 8 Oct research rule, in units (favourites staked to win 1 unit,
                           # underdogs just under 1 unit; not a decided staking rule). None: no stake column

if Path(sys.executable).resolve() != tp.config.interpreter("tennis").resolve():
    print(f"WARNING: this kernel runs {sys.executable}; select the `tennis` conda env as the notebook's kernel "
          f"({tp.config.interpreter('tennis')}). The steps themselves always run in their own environments.")

In [ ]:
# STATUS: the scheduled downloads, each source's freshness, the model instance and its fits, checkpoints, recent runs, disk
tp.status();

---
## A. Bet day (run top to bottom)
A2 to A5 take about 10 minutes when the checkpoint and the TabPFN caches are warm, plus about 2 minutes for A3b's data check (seconds when nothing new was downloaded). The first run after Tennis My Life revises past rows recomputes more feature state, which can add 10–15 minutes.

**The flow:** fresh data (A1) → the matches Pinnacle lists (A2) → the matchups (A3) → **the data check (A3b): what the run's gate will do to each match, and why** → the prediction run (A4) → **the bet sheet (A5)** → reprice when prices move (A6).

**Since 14 Oct 2026 a missing box score never blocks a bet** ([decision](../docs/decisions/2026-10-14-box-score-gate.md)): of a main-tour, Challenger or qualifying match it makes a bet need a 1-point edge; of a team event or an ITF match it is only noted. Only an unknown indoor input, or a qualifier with no qualifying match in the data, still blocks.

**Since 22 Oct 2026 the qualifying of the 250s and above is predicted and bet too, with the same live model** ([decision](../docs/decisions/2026-10-22-qualifying-bets.md)). A2 finds each qualifying match's event (a round 1's from the event's previous edition) and its round (from TennisLive's results). A round 2 or 3 is blocked while a player's round before is missing from the data: a man's until Tennis My Life publishes it (run A1 again), a woman's until the weekly lower-level feed brings it.

In [ ]:
# A1 MORNING REFRESH: box scores first. The 07:30 task normally did it; this runs whatever of today's is missing and
#    shows any held Tennis My Life file (kept out until reviewed: B2). The prediction run downloads results again itself.
tp.morning_refresh(dry_run=DRY_RUN)

In [ ]:
# A2 TODAY'S MATCHES: Pinnacle's listing now (3 paced requests; raw copies and receipts kept; a pull under a minute after
#    the last reuses it), each match linked to Tennis My Life's event and to the players Stage 0 knows.
#    status "ok" = predictable; every other row says why it is left out (Challenger/ITF/doubles/in-play, a first round
#    before the event's first played match, a name that names two Stage 0 players, already played, ...).
#    Qualifying of the 250s and above (since 22 Oct 2026): its round is read from TennisLive's results of the event (a
#    few paced requests, receipts kept); a round 1 before anything of its edition is published takes the event from
#    its previous edition. A round TennisLive cannot confirm is left out with the reason: run A2 again later.
tp.record_pinnacle(dry_run=DRY_RUN)
matches = tp.todays_matches(hours_ahead=HOURS_AHEAD)
tp.show_matches(matches)

In [ ]:
# A3 THE MATCHUPS CSV of the linked matches (player_a = Pinnacle's home player, p_a = his margin-free chance; the quote
#    time and Pinnacle's odds ride along), then the daily pipeline's own refusals checked before the expensive run.
MATCHUPS = tp.build_matchups(matches)                 # only=["Sinner"] / drop=["Rune"] to choose matches
tp.check_matchups(MATCHUPS)

# Fallback for a match Pinnacle does not list (BetOnline's line stands in only while Pinnacle has none):
# MATCHUPS = tp.manual_matchups([dict(tour="atp", tourney_id="2026-5014", round="R16", player_a="Jannik Sinner",
#                                     player_b="Holger Rune", day="2026-10-12", start_utc="2026-10-12T08:00:00Z",
#                                     odds_a=-250, odds_b=210)], source="betonline")

In [ ]:
# A3b DATA CHECK, before the 10-minute run (about 2 minutes; seconds when nothing new was downloaded): the run's gate
#    applied now to today's data, the same Stage 0 and lower-level feeds the run will use. Each match is "ready",
#    "ready once filled" (the run fills the box score from an official feed: a woman's from the WTA's, a man's main-draw
#    match from the ATP's), "needs a 1-point edge" (a main-tour, Challenger or qualifying box score is missing) or
#    "will be blocked" (a qualifier without a qualifying match; a qualifying round 2 or 3 whose player's round before is
#    not in the data yet), with the cause (Tennis My Life has none yet / our consistency check rejected Tennis My
#    Life's / ...) and what fixes it. Team-event and ITF gaps are only noted.
#    Waiting helps the first cause: run A1 later, then this cell again. The run downloads once more and decides with
#    its own gate.
CHECK = tp.data_check(MATCHUPS, dry_run=DRY_RUN)
# MATCHUPS = tp.build_matchups(matches, drop=["Ruud"])     # leave out a match that will be blocked, then A4

In [ ]:
# A4 PREDICT with MODEL: the template check (the active instance is the decided recipe, fitted for the season), then the
#    daily pipeline's run_day --refresh in a new run folder (box scores refreshed, today's tables in a sandbox, the 57
#    inputs from the checkpoint, the gate, the 16 seeds at 91 prices, stage 2). About 7 minutes warm.
#    MODEL "tabpfn35_thinking" (SPEND_TOKENS = True): after the local fit, one paid TabPFN-3.5-Thinking call on the same
#    rows (about a minute); a new instance's first Thinking run makes its paid season fit first (about 8 minutes). The
#    prediction used is the 50/50 average of the two models. If the Thinking call fails, the run's predictions are the
#    local model's alone, and this cell and the bet sheet say so.
RUN = tp.predict_day(MATCHUPS, model=MODEL, spend_tokens=SPEND_TOKENS, dry_run=DRY_RUN)    # None in a dry run

In [ ]:
# A5 THE BET SHEET, shown here and saved as predictions/<day>/bet_sheet_<time>.xlsx (open it in Excel). One row per
#    match, bets first: BET (the player, the stake, Pinnacle's US odds and the lowest US odds still worth taking), PASS
#    (no side is +EV at Pinnacle's odds, or, with a missing main-tour box score, not by the 1-point edge it asks; the
#    odds that would make it a bet), BLOCKED (an unknown indoor input, a qualifier without a qualifying match, a
#    qualifying round whose round before is missing, a stale price, a started match), each with the model's and the
#    market's chance and a plain note. All odds are American. Qualifying matches show as Q1, Q2 or Q3 in Event.
#    With MODEL "tabpfn35_thinking": the non-thinking, the thinking and the ensemble chance without patterns side by
#    side; the model chance, edge, expected profit and decision are the ensemble's (the 50/50 average, plus the
#    patterns). The workbook adds every figure of both players (Matches), the listed matches not predicted, the run's
#    provenance and a glossary (How to read). The machine record behind each sheet is in predictions/<day>/records/.
diag = tp.bet_diagnostics(RUN, stale_minutes=STALE_MINUTES, stake_rule=STAKE_RULE) if RUN else None

# At another venue too (its prices given as decimal or American odds, or a Polymarket share price), with its fee on net
# winnings:
# diag = tp.bet_diagnostics(RUN, venue_prices={"Jannik Sinner": 0.72, "Holger Rune": 0.30}, venue="polymarket", venue_fee=0.0)

In [ ]:
# A6 PRICES MOVED: a fresh Pinnacle pull, each new price read off the run's stored curve (no refit; seconds), then a new
#    bet sheet at the new prices (the earlier one is kept). A price outside the curve's 0.05-0.95 needs a new run (A4).
if RUN:
    REPRICE = tp.reprice(RUN, dry_run=DRY_RUN)
    if REPRICE:
        diag = tp.bet_diagnostics(RUN, reprice_file=REPRICE, stale_minutes=STALE_MINUTES, stake_rule=STAKE_RULE)

In [ ]:
# A7 GRADE an earlier run once Stage 0 holds its results (after B6). Its numbers decide nothing.
# tp.grade(RUN)
# RUN = tp.latest_run()          # the newest live run, e.g. after restarting the kernel

---
## B. Data update (weekly, and before a rebuild)
These cells run the project's own downloaders, with their pacing, receipts and stop-on-block rules. B7 runs B2–B5 in one go.

In [ ]:
# B1 THE DAILY DOWNLOAD TASK NOW: every job due today (Tennis My Life, serve speeds, ITF; the canary and TDI on Mondays).
#    only=["tml", "speeds"] to choose; plan_only=True lists the due jobs and stops.
tp.scheduled_downloads(dry_run=DRY_RUN);

In [ ]:
# B2 RESULTS AND BOX SCORES (Tennis My Life), and the files it holds for review (rows vanished, a column lost, ...).
tp.refresh_results(dry_run=DRY_RUN);
print("held:", tp.held_tml_files())
# After reviewing a held file against the current copy. Lower-level files (Challenger, qualifying) then need the
# candidate route before the model may use them:
# tp.accept_held_tml("2025_challenger.csv@78b86009de42")

In [ ]:
# B3 ATP SERVE SPEEDS (stage 2 reads each player's 28-day form). After missed days, raise days_back.
tp.refresh_speeds(days_back=1, dry_run=DRY_RUN);

In [ ]:
# B4 TIMELINE SOURCES: exact match days (TDI's last ten days, about 20 minutes; the WTA API's recent editions, restaged).
tp.refresh_timeline_sources(dry_run=DRY_RUN);

In [ ]:
# B5 ODDS HISTORY: tennis-data.co.uk's workbooks and OddsTrader's lines (last month and this month, fetched again).
tp.refresh_odds_history(dry_run=DRY_RUN);

In [ ]:
# B6 REBUILD THE SHARED TABLES (Stage 0, Stage 1's odds, the timeline, the leak-proof features): about 50 minutes.
#    Other sessions read these tables: check none is running, then set confirm=True. Needed before a rebuild (C) and for
#    grading (A7); a bet day does not need it.
tp.rebuild_shared_tables(confirm=False, dry_run=DRY_RUN);

In [ ]:
# B7 ALL OF B2-B5 IN ONE CALL (rebuild=True, confirm_rebuild=True adds B6)
# tp.weekly_update(dry_run=DRY_RUN)

In [ ]:
# B8 THE LOWER-LEVEL FEED AND THE COVERAGE ALARM (since 13 Oct 2026). Tennis My Life publishes no ITF, WTA 125, women's
#    qualifying or men's Challenger qualifying; those levels reach every build and prediction run through the feed, a
#    weekly snapshot built from TennisLive, the WTA API, the ITF's live data and TDI. A second Windows task runs it by
#    itself (TennisLive every day at 12:00; the WTA API, the feed and the coverage check on Wednesdays). The alarm lists
#    any level whose matches or box scores fell below 70% of the same months one and two years earlier.
print(tp.feed_status())
# tp.lower_level_feed(only=["tennislive"], dry_run=DRY_RUN)     # all four jobs: about an hour (the feed ~50 minutes)

---
## C. Rebuild and retrain (new season, or after accepted data)
"The current best model" is a recipe built on the day's tables (`docs/decisions/2026-10-08-current-best-model.md`). An **instance** is one build of it: the model frame, the season fits, and the build's sandbox, which the daily pipeline reads. A build whose lower-level history differs from the frozen reference is a **candidate**: build it with `candidate=True`, compare it in C5, and accepting it is the owner's call.

In [ ]:
# C0 RETRAIN, when you say it's time (your rule of 20 Oct 2026): the template built on the shared tables as last rebuilt,
#    its live fit trained through the cut-off (this season's matches as training rows) and stage 2 grown with it, the
#    harm check against the instance in use, and the switch only with no harm (the replay proof; rollback on failure).
#    About 5-6 hours, detached: it survives the kernel and VS Code. First B5, B2 and B4, then B6 with confirm=True
#    (after Wednesday's lower-level feed). The first A4 after it builds the new TabPFN caches (about 15 minutes, once)
#    and, with Thinking on, makes Thinking's paid fit on the same rows (about 180k tokens).
tp.show_model_age()
PLAN = tp.retrain()                     # the plan and its checks; nothing starts
# PLAN = tp.retrain(confirm=True)       # start it

In [ ]:
# C1 INSTANCES: what exists, what the active one holds, and whether it is ready to predict this season
print(tp.list_instances())
tp.instance_status()
tp.template_check();

In [ ]:
# C2 BUILD A NEW INSTANCE (about 3 hours, no GPU): the chain in its own sandbox. Keep the sandbox afterwards. For a
#    retrain, C0 runs C2-C8 by itself; these cells are the same steps one at a time.
#    It needs about 90 GB free: the sandbox (~16 GB: since 20 Oct the daily pipeline's TabPFN caches are left out of
#    its copy) and the new instance's own caches (~55 GB at its first prediction).
INSTANCE = tp.new_instance_path()          # data/studies/current_best_model/template_<today>
tp.build_instance(INSTANCE, candidate=False, dry_run=True);         # dry_run=True: shows the 8 steps; set DRY_RUN to run
# resume after a failure: tp.build_instance(INSTANCE, start_at="lane3", dry_run=DRY_RUN)

In [ ]:
# C3 FINISH IT: the decided odds input "Pinnacle first" (OddsTrader's every-book lines linked) and the 57-input list
tp.finish_instance(INSTANCE, dry_run=True);

In [ ]:
# C4 SEASON FITS with MODEL. tabpfn35: local GPU, seeds 0-15 (about an hour for 2019-2026; waits its turn on the card).
#    tabpfn35_thinking: Thinking's hosted fits only, one per season, a FREE quote unless spend_tokens=True (A4 makes the
#    current season's by itself; the local fits still come from tabpfn35).
tp.fit_seasons(INSTANCE, model=MODEL, seasons="2019-2026", seeds="0-15", dry_run=True);

In [ ]:
# C5 A CANDIDATE AGAINST THE FROZEN REFERENCE (compare.py): read the input check first ("uneven" is the warning sign),
#    then both stages; then the adoption framework (adoption_read.py). Accepting it is the owner's call.
# tp.compare_candidate(INSTANCE)
# tp.check_history()        # the shared Stage 0 against the reference: gated (lower-level) and allowed changes

In [ ]:
# C6 POINT THE DAILY PIPELINE AT AN INSTANCE (checks it, then lists the steps; the switch itself needs the daily
#    pipeline's DP_INSTANCE, requested from its session)
# tp.use_instance(INSTANCE, tree="C:/Users/zerom/tnr_build_<YYYYMMDD>")     # tp.use_instance(None): back to the default

In [ ]:
# C7 A CHECKPOINT: every feature family's state before a day (about 11 minutes). Live runs leave new ones by themselves.
# tp.make_checkpoint("2026-10-05", dry_run=DRY_RUN)

In [ ]:
# C8 THE REPLAY PROOF: a past day replayed through the daily pipeline must reproduce the instance's 57 inputs exactly
#    (about 7 minutes with the GPU); then the diagnostics of that replay
# REPLAY = tp.replay_check(dry_run=DRY_RUN)

In [ ]:
# C9 TABPFN-3.5-THINKING's SEASON FIT: a FREE quote of this season's fit and of a daily call, with the account's
#    remaining quota. A4 with MODEL "tabpfn35_thinking" and SPEND_TOKENS = True makes the fit by itself when the active
#    instance has none for the season; spend_tokens=True here makes it ahead of a bet day (owner approval).
# tp.fit_seasons(model="tabpfn35_thinking", seasons="2026")
# tp.thinking_status(tp.active_instance())

**Season rollover** (each new season, before its first prediction; CLAUDE.md, "The level layer"): refit the level layer walk-forward into a new folder (`relationship.py --method F4`), re-issue its config with the new path and hash, and re-run the identity and strength checks. Then build the season's instance (C2–C4) and fit the new season. The odds sources take every season tennis-data.co.uk holds (no last season since 12 Oct 2026), so a new season's matches get the Pinnacle-first odds input, not the books' average, once B5 has fetched its tennis-data file and OddsTrader's months. The level layer's refit is another study's code, so the owner decides.

---
## Maintenance

In [ ]:
# M1 WHAT IS RUNNING, WHAT RAN: steps still running (e.g. a build left running when VS Code closed: its log keeps
#    growing), the latest receipts, jobs paused after a block, and the tnr_* trees on disk (sizes=True is slow)
print(tp.running())
for r in tp.receipts(last=10):
    print(r["started"], r["status"], r["step"], r["log"])
print("paused:", tp.paused_jobs())
for t in tp.sandboxes(sizes=False):
    print(t)
# A prediction run stopped midway leaves its work tree (C:/Users/zerom/tnr_daily_nb, ~18 GB). Remove it with the daily
# pipeline's own cleanup, never by hand: it holds junctions to the repository's data/raw and data/dump.
# tp.cleanup_work_tree()